In [ ]:
import pandas as pd
import numpy as np
import warnings
import pickle
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import timedelta

import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import shap

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH  = "site_day_panel.csv"   # update path if needed
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Window sizes ──────────────────────────────────────────────────────────────
LOOKBACK_DAYS = 182   # ~6 months of history used as input
HORIZON_DAYS  = 182   # ~6 months ahead to predict
STEP_DAYS     = 30    # slide training window every 30 days
MIN_TRAIN_ROWS = 50   # minimum positive samples to train Model C

print("✅ Imports and config loaded")
print(f"   Lookback : {LOOKBACK_DAYS} days")
print(f"   Horizon  : {HORIZON_DAYS} days")
print(f"   Step     : {STEP_DAYS} days")


In [ ]:
WEATHER_FEATURES = [
    "t2m", "t2m_min", "t2m_max", "t2mdew",
    "t2m_rolling_mean_7d", "t2m_rolling_mean_14d", "t2m_rolling_mean_30d",
    "t2m_min_rolling_mean_7d", "t2m_min_rolling_mean_30d",
    "t2m_max_rolling_mean_7d", "t2m_max_rolling_mean_30d",
    "t2mdew_rolling_mean_7d", "t2mdew_rolling_mean_30d",
    "uv_index_rolling_mean_7d", "uv_index_rolling_mean_14d",
    "cloud_rolling_mean_7d", "cloud_rolling_mean_14d",
    "precip_rolling_sum_7d", "precip_rolling_sum_14d", "precip_rolling_sum_30d",
    "dry_day_count_7d", "dry_day_count_14d", "dry_day_count_30d",
    "day_of_year_sin", "day_of_year_cos", "warm_season_flag", "month",
]

CYANO_FEATURES = [
    "latest_overall_chlorophyll_ugl_mean",
    "latest_overall_phycocyanin_ugl_mean",
    "latest_overall_phycocyanin_to_chlorophyll_ratio_mean",
    "latest_overall_water_temperature_c",
    "latest_overall_ph",
    "latest_overall_dissolved_oxygen_mg_l",
    "latest_overall_turbidity_ntu",
    "delta_7d_overall_chlorophyll_ugl_mean",
    "delta_14d_overall_chlorophyll_ugl_mean",
    "delta_7d_overall_phycocyanin_ugl_mean",
    "delta_14d_overall_phycocyanin_ugl_mean",
    "delta_7d_overall_phycocyanin_to_chlorophyll_ratio_mean",
    "days_since_last_sample",
]

print(f"Weather features : {len(WEATHER_FEATURES)}")
print(f"Cyano features   : {len(CYANO_FEATURES)}")


In [ ]:
def load_data(path):
    df = pd.read_csv(path, low_memory=False, parse_dates=["date"])
    df = df.sort_values(["site_name", "date"]).reset_index(drop=True)
    le = LabelEncoder()
    df["site_id"]     = le.fit_transform(df["site_name"])
    df["category_id"] = (df["major_category"] == "Category_2").astype(int)
    return df, le

df, site_encoder = load_data(DATA_PATH)

print(f"Shape            : {df.shape}")
print(f"Date range       : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Sites            : {df['site_name'].nunique()}")
print(f"Advisory-active rows : {df['advisory_active'].sum():,} / {len(df):,}  "
      f"({df['advisory_active'].mean():.1%})")


In [ ]:
# Advisory event summary
active = df[df["advisory_active"] == 1].copy()
events = (active
    .groupby(["site_name", "advisory_event_id"])["date"]
    .agg(["min", "max"])
    .reset_index()
)
events.columns = ["site_name", "event_id", "start", "end"]
events["duration_days"] = (events["end"] - events["start"]).dt.days + 1

print(f"Total advisory events : {len(events)}")
print(f"Duration — min: {events['duration_days'].min()}d  "
      f"median: {events['duration_days'].median():.0f}d  "
      f"max: {events['duration_days'].max()}d")
events.sort_values("start")


In [ ]:
# Advisory activity by site
fig, ax = plt.subplots(figsize=(12, 4))
for i, (site, grp) in enumerate(df.groupby("site_name")):
    active_dates = grp[grp["advisory_active"] == 1]["date"]
    ax.scatter(active_dates, [i] * len(active_dates), s=1, color="crimson", alpha=0.5)
ax.set_yticks(range(df["site_name"].nunique()))
ax.set_yticklabels(df["site_name"].unique())
ax.set_xlabel("Date")
ax.set_title("Advisory Active Days by Site (red = active)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "advisory_timeline.png", dpi=120)
plt.show()


In [ ]:
def _days_since_last_onset(window):
    onset_idx = window.index[window["onset_label"] == 1].tolist()
    if not onset_idx:
        return LOOKBACK_DAYS  # sentinel: no onset in window
    last_pos = window.index.get_loc(onset_idx[-1])
    return len(window) - 1 - last_pos


def build_window_features(window: pd.DataFrame) -> dict:
    """Compress a ~182-day lookback window into a flat feature vector."""
    feats = {}

    # Weather features: last / mean / trend
    for col in WEATHER_FEATURES:
        if col not in window.columns:
            continue
        vals = window[col].dropna()
        if len(vals) == 0:
            feats[f"{col}_last"] = feats[f"{col}_mean"] = feats[f"{col}_trend"] = np.nan
        else:
            feats[f"{col}_last"] = vals.iloc[-1]
            feats[f"{col}_mean"] = vals.mean()
            half = len(vals) // 2
            feats[f"{col}_trend"] = (vals.iloc[half:].mean() - vals.iloc[:half].mean()
                                     if half > 0 else 0.0)

    # Cyano / monitoring features: last / max / mean / recent-30d mean
    for col in CYANO_FEATURES:
        if col not in window.columns:
            continue
        vals = window[col].dropna()
        if len(vals) == 0:
            feats[f"{col}_last"] = feats[f"{col}_max"] = np.nan
            feats[f"{col}_mean"] = feats[f"{col}_recent"] = np.nan
        else:
            feats[f"{col}_last"]   = vals.iloc[-1]
            feats[f"{col}_max"]    = vals.max()
            feats[f"{col}_mean"]   = vals.mean()
            feats[f"{col}_recent"] = window[col].dropna().iloc[-30:].mean()

    # Advisory history in the lookback window
    feats["advisory_days_in_window"]     = window["advisory_active"].sum()
    feats["advisory_frac_in_window"]     = window["advisory_active"].mean()
    feats["advisory_active_last_day"]    = int(window["advisory_active"].iloc[-1])
    feats["days_since_advisory_start"]   = _days_since_last_onset(window)
    feats["n_advisory_events_in_window"] = window["onset_label"].sum()

    # Seasonality at prediction point (last row of window)
    last_row = window.iloc[-1]
    feats["month_at_prediction"]      = last_row["month"]
    feats["doy_sin_at_prediction"]    = last_row["day_of_year_sin"]
    feats["doy_cos_at_prediction"]    = last_row["day_of_year_cos"]
    feats["warm_season_at_prediction"]= last_row["warm_season_flag"]

    # Site identity
    feats["site_id"]     = int(last_row["site_id"])
    feats["category_id"] = int(last_row["category_id"])

    return feats

print("✅ Feature engineering functions defined")


In [ ]:
LABEL_COLS = [
    "site_name", "window_end_date", "onset_in_horizon",
    "currently_active", "days_to_onset",
    "advisory_days_in_horizon", "advisory_frac_in_horizon",
]


def build_training_set(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for site, grp in df.groupby("site_name"):
        grp = grp.sort_values("date").reset_index(drop=True)
        n   = len(grp)
        for start_idx in range(LOOKBACK_DAYS, n - HORIZON_DAYS, STEP_DAYS):
            lookback = grp.iloc[start_idx - LOOKBACK_DAYS : start_idx]
            horizon  = grp.iloc[start_idx : start_idx + HORIZON_DAYS]

            currently_active = int(lookback.iloc[-1]["advisory_active"])
            new_onsets       = horizon[horizon["onset_label"] == 1]

            if currently_active:
                onset_in_horizon = 0
                days_to_onset    = -1
            else:
                onset_in_horizon = int(len(new_onsets) > 0)
                days_to_onset    = (int(new_onsets.iloc[0].name - horizon.index[0])
                                    if onset_in_horizon else HORIZON_DAYS)

            advisory_days  = int(horizon["advisory_active"].sum())
            advisory_frac  = advisory_days / HORIZON_DAYS

            feats = build_window_features(lookback)
            feats.update({
                "site_name"                 : site,
                "window_end_date"           : lookback.iloc[-1]["date"],
                "onset_in_horizon"          : onset_in_horizon,
                "currently_active"          : currently_active,
                "days_to_onset"             : days_to_onset,
                "advisory_days_in_horizon"  : advisory_days,
                "advisory_frac_in_horizon"  : advisory_frac,
            })
            records.append(feats)
    return pd.DataFrame(records)


train_df = build_training_set(df)
print(f"Training windows generated : {len(train_df)}")
print(f"Feature dimensions         : {len([c for c in train_df.columns if c not in LABEL_COLS])}")

# Label breakdown
non_active = train_df[train_df["currently_active"] == 0]
print(f"Onset rate (non-active)    : {non_active['onset_in_horizon'].mean():.2%}")
print(f"Mean advisory days         : {train_df['advisory_days_in_horizon'].mean():.1f}")


In [ ]:
# Label distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

non_active["onset_in_horizon"].value_counts().plot(
    kind="bar", ax=axes[0], color=["steelblue", "crimson"],
    title="Model A — Onset in Horizon\n(non-active windows only)"
)
axes[0].set_xticklabels(["No onset (0)", "Onset (1)"], rotation=0)
axes[0].set_ylabel("Count")

train_df["advisory_days_in_horizon"].hist(bins=30, ax=axes[1], color="teal", edgecolor="white")
axes[1].set_title("Model B — Advisory Days in Horizon")
axes[1].set_xlabel("Days")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "label_distributions.png", dpi=120)
plt.show()


---
## 6. LightGBM Hyperparameters

In [ ]:
LGB_BASE_PARAMS = dict(
    n_estimators     = 400,
    learning_rate    = 0.05,
    num_leaves       = 31,
    max_depth        = 6,
    min_child_samples= 10,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 0.1,
    random_state     = 42,
    n_jobs           = -1,
    verbose          = -1,
)

FEATURE_COLS = [c for c in train_df.columns if c not in LABEL_COLS]
X_all = train_df[FEATURE_COLS]

print(f"Feature matrix shape : {X_all.shape}")
print(f"LightGBM params      : {LGB_BASE_PARAMS}")


---
## 7. Temporal Cross-Validation Helper

Uses `TimeSeriesSplit` so validation folds are always **future** data.  
This prevents data leakage that standard k-fold would introduce.


In [ ]:
def temporal_cv(X, y, task="classification", n_splits=4, label=""):
    """
    Run time-series cross-validation and return per-fold scores.
    task : 'classification' → AUC-ROC
           'regression'     → MAE
    """
    tscv   = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X), 1):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        if task == "classification":
            m = lgb.LGBMClassifier(**LGB_BASE_PARAMS)
        else:
            m = lgb.LGBMRegressor(**LGB_BASE_PARAMS)

        m.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(period=-1),
            ],
        )

        if task == "classification":
            pred = m.predict_proba(X_val)[:, 1]
            s    = roc_auc_score(y_val, pred) if y_val.nunique() > 1 else 0.5
            metric_name = "AUC-ROC"
        else:
            pred = m.predict(X_val)
            s    = mean_absolute_error(y_val, pred)
            metric_name = "MAE"

        scores.append(s)
        print(f"  {label} Fold {fold}: {metric_name} = {s:.4f}")

    print(f"  {label} → Mean {metric_name}: {np.mean(scores):.4f}  ±{np.std(scores):.4f}")
    return scores

print("✅ CV helper defined")


---
## 8. Train the Three Models

### 8A. Model A — Onset Classifier

Binary classification: *Will a new advisory start in the next 6 months?*  
Trained **only on non-active windows** (currently_active == 0).


In [ ]:
mask_a = train_df["currently_active"] == 0
X_a    = X_all[mask_a]
y_a    = train_df.loc[mask_a, "onset_in_horizon"]

print(f"Model A — training rows : {len(X_a)}")
print(f"Positive rate           : {y_a.mean():.3f}  ({y_a.sum()} positives)")
print()

scores_a = temporal_cv(X_a, y_a, task="classification", label="Model A")


In [ ]:
# Final fit on all data
clf_a = lgb.LGBMClassifier(**LGB_BASE_PARAMS)
clf_a.fit(X_a, y_a, callbacks=[lgb.log_evaluation(period=-1)])
print("✅ Model A trained")


### 8B. Model B — Duration Regressor

Regression: *How many total advisory days will occur in the horizon?*  
Trained on **all windows** (including currently-active).

In [ ]:
X_b = X_all
y_b = train_df["advisory_days_in_horizon"].astype(float)

print(f"Model B — training rows  : {len(X_b)}")
print(f"Mean advisory days       : {y_b.mean():.1f}  |  Max: {y_b.max()}")
print()

scores_b = temporal_cv(X_b, y_b, task="regression", label="Model B")


In [ ]:
reg_b = lgb.LGBMRegressor(**LGB_BASE_PARAMS)
reg_b.fit(X_b, y_b, callbacks=[lgb.log_evaluation(period=-1)])
print("✅ Model B trained")


### 8C. Model C — Timing Regressor

Regression: *How many days from now until advisory onset?*  
Trained only on windows where an onset **did** occur in the horizon.

In [ ]:
mask_c = (train_df["currently_active"] == 0) & (train_df["onset_in_horizon"] == 1)
X_c    = X_all[mask_c]
y_c    = train_df.loc[mask_c, "days_to_onset"].astype(float)

print(f"Model C — training rows  : {len(X_c)}")
print(f"Mean days to onset       : {y_c.mean():.1f}  |  Std: {y_c.std():.1f}")
print()

if len(X_c) >= MIN_TRAIN_ROWS:
    scores_c = temporal_cv(X_c, y_c, task="regression", label="Model C")
    reg_c = lgb.LGBMRegressor(**LGB_BASE_PARAMS)
    reg_c.fit(X_c, y_c, callbacks=[lgb.log_evaluation(period=-1)])
    print("✅ Model C trained")
else:
    print(f"⚠️  Only {len(X_c)} positive samples — using median fallback")
    reg_c = None
    scores_c = [None]


---
## 9. Cross-Validation Summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

folds = [1, 2, 3, 4]
colors = ["#2196F3", "#4CAF50", "#FF5722"]

axes[0].plot(folds, scores_a, "o-", color=colors[0], lw=2)
axes[0].axhline(np.mean(scores_a), ls="--", color=colors[0], alpha=0.5)
axes[0].set_title(f"Model A — Onset Classifier\nMean AUC-ROC: {np.mean(scores_a):.4f}")
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("AUC-ROC")
axes[0].set_ylim(0.5, 1.0)

axes[1].plot(folds, scores_b, "o-", color=colors[1], lw=2)
axes[1].axhline(np.mean(scores_b), ls="--", color=colors[1], alpha=0.5)
axes[1].set_title(f"Model B — Duration Regressor\nMean MAE: {np.mean(scores_b):.1f} days")
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("MAE (days)")

if scores_c[0] is not None:
    axes[2].plot(folds, scores_c, "o-", color=colors[2], lw=2)
    axes[2].axhline(np.mean(scores_c), ls="--", color=colors[2], alpha=0.5)
    axes[2].set_title(f"Model C — Timing Regressor\nMean MAE: {np.mean(scores_c):.1f} days")
    axes[2].set_xlabel("Fold")
    axes[2].set_ylabel("MAE (days)")
else:
    axes[2].text(0.5, 0.5, "Insufficient data", ha="center", va="center",
                 transform=axes[2].transAxes, fontsize=12, color="gray")
    axes[2].set_title("Model C — Skipped")

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cv_scores.png", dpi=120)
plt.show()


---
## 10. Feature Importance (Model A — Onset Classifier)

In [ ]:
importance = (pd.DataFrame({
        "feature"   : FEATURE_COLS,
        "importance": clf_a.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .head(25)
)

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=importance, y="feature", x="importance", palette="Blues_r", ax=ax)
ax.set_title("Top 25 Feature Importances — Model A (Onset Classifier)")
ax.set_xlabel("Importance (split count)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_importance.png", dpi=120)
plt.show()

importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
importance


---
## 11. SHAP Explainability (Model A)

SHAP values show the **contribution of each feature** to the model's predictions,  
giving a principled measure of feature impact beyond simple split counts.


In [ ]:
explainer   = shap.TreeExplainer(clf_a)
shap_values = explainer.shap_values(X_a)

# For binary classifier, shap_values may be a list [neg_class, pos_class]
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(10, 7))
shap.summary_plot(sv, X_a, feature_names=FEATURE_COLS,
                  max_display=20, show=False)
plt.title("SHAP Summary — Model A (Onset Classifier)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "shap_summary.png", dpi=120)
plt.show()
print("SHAP plot saved")


---
## 12. Prediction Function

`predict_next_6_months(site_name, prediction_date)` returns:

| Key | Description |
|-----|-------------|
| `advisory_probability` | P(advisory occurs in horizon) |
| `confidence_label` | High / Moderate / Low / Unlikely |
| `predicted_advisory_days` | Estimated total advisory days |
| `predicted_start_date` | Estimated advisory start |
| `predicted_end_date` | Estimated advisory end |


In [ ]:
def predict_next_6_months(site_name: str, prediction_date=None):
    """
    Predict advisory activity for the 6 months following prediction_date.
    
    Parameters
    ----------
    site_name       : Name of the site (must match site_name in dataset)
    prediction_date : str or datetime. Defaults to last available date.
    
    Returns
    -------
    dict with prediction results
    """
    site_df = df[df["site_name"] == site_name].sort_values("date").reset_index(drop=True)

    if prediction_date is None:
        prediction_date = site_df["date"].max()
    else:
        prediction_date = pd.to_datetime(prediction_date)

    start_date = prediction_date - timedelta(days=LOOKBACK_DAYS)
    lookback   = site_df[
        (site_df["date"] >= start_date) & (site_df["date"] <= prediction_date)
    ].copy()

    if len(lookback) < 30:
        return {"error": f"Only {len(lookback)} days of lookback data (need ≥30)"}

    feats   = build_window_features(lookback)
    feat_df = pd.DataFrame([feats])[FEATURE_COLS]

    currently_active = feats["advisory_active_last_day"]

    # ── Advisory probability ─────────────────────────────────────────────────
    if currently_active:
        adv_prob = 0.92   # already in advisory — very likely to continue
    else:
        adv_prob = float(clf_a.predict_proba(feat_df)[0, 1])

    # ── Duration ─────────────────────────────────────────────────────────────
    pred_days = float(np.clip(reg_b.predict(feat_df)[0], 0, HORIZON_DAYS))

    # ── Timing ───────────────────────────────────────────────────────────────
    horizon_start = prediction_date + timedelta(days=1)

    if currently_active:
        pred_start    = horizon_start
        days_to_onset = 0
    elif adv_prob > 0.30 and reg_c is not None:
        days_to_onset = float(np.clip(reg_c.predict(feat_df)[0], 0, HORIZON_DAYS - 1))
        pred_start    = horizon_start + timedelta(days=int(days_to_onset))
    else:
        pred_start    = None

    if pred_start is not None and pred_days > 0:
        pred_end = min(pred_start + timedelta(days=int(pred_days)),
                       horizon_start + timedelta(days=HORIZON_DAYS))
    else:
        pred_end = None

    # ── Confidence label ─────────────────────────────────────────────────────
    conf = ("High"     if adv_prob >= 0.75 else
            "Moderate" if adv_prob >= 0.45 else
            "Low"      if adv_prob >= 0.20 else
            "Unlikely")

    return {
        "site_name"               : site_name,
        "prediction_date"         : prediction_date.strftime("%Y-%m-%d"),
        "horizon_start"           : horizon_start.strftime("%Y-%m-%d"),
        "horizon_end"             : (horizon_start + timedelta(days=HORIZON_DAYS)).strftime("%Y-%m-%d"),
        "advisory_probability"    : round(adv_prob, 4),
        "confidence_label"        : conf,
        "predicted_advisory_days" : int(round(pred_days)),
        "predicted_start_date"    : pred_start.strftime("%Y-%m-%d") if pred_start else None,
        "predicted_end_date"      : pred_end.strftime("%Y-%m-%d")   if pred_end   else None,
        "currently_active"        : bool(currently_active),
        "lookback_days_available" : len(lookback),
    }

print("✅ predict_next_6_months() defined")


---
## 13. Demo Predictions — All Sites (Latest Date)

In [ ]:
results = []
for site in df["site_name"].unique():
    pred = predict_next_6_months(site)
    results.append(pred)

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / "predictions_latest.csv", index=False)

# Display formatted
display_cols = [
    "site_name", "advisory_probability", "confidence_label",
    "predicted_advisory_days", "predicted_start_date", "predicted_end_date",
    "currently_active",
]
results_df[display_cols].style.background_gradient(
    subset=["advisory_probability"], cmap="RdYlGn_r"
)


In [ ]:
# Probability bar chart by site
fig, ax = plt.subplots(figsize=(10, 5))
colors  = ["crimson" if p >= 0.75 else "orange" if p >= 0.45 else "steelblue"
           for p in results_df["advisory_probability"]]
bars = ax.barh(results_df["site_name"], results_df["advisory_probability"], color=colors)
ax.axvline(0.75, ls="--", color="crimson",    alpha=0.6, label="High threshold (0.75)")
ax.axvline(0.45, ls="--", color="orange",     alpha=0.6, label="Moderate threshold (0.45)")
ax.axvline(0.20, ls="--", color="steelblue",  alpha=0.4, label="Low threshold (0.20)")
ax.set_xlabel("Advisory Probability (next 6 months)")
ax.set_title("Predicted Advisory Probability by Site")
ax.set_xlim(0, 1)
ax.legend(loc="lower right")
for bar, p in zip(bars, results_df["advisory_probability"]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{p:.1%}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "predictions_chart.png", dpi=120)
plt.show()


---
## 14. Single-Site Deep Dive

Change `SITE` and `PRED_DATE` to run a prediction for any site / date.


In [ ]:
SITE      = "Spectacle Pond"   # ← change me
PRED_DATE = None               # ← change to e.g. "2024-06-01", or None for latest

result = predict_next_6_months(SITE, PRED_DATE)

print(f"Site              : {result['site_name']}")
print(f"Prediction date   : {result['prediction_date']}")
print(f"Forecast window   : {result['horizon_start']}  →  {result['horizon_end']}")
print()
print(f"Advisory prob     : {result['advisory_probability']:.1%}  [{result['confidence_label']}]")
print(f"Predicted days    : {result['predicted_advisory_days']}")
print(f"Est. start date   : {result['predicted_start_date'] or 'N/A'}")
print(f"Est. end date     : {result['predicted_end_date']   or 'N/A'}")
print(f"Currently active  : {result['currently_active']}")
print(f"Lookback data pts : {result['lookback_days_available']}")


---
## 15. Save Models

In [ ]:
model_bundle = {
    "onset_clf"   : clf_a,
    "duration_reg": reg_b,
    "timing_reg"  : reg_c,
    "feature_cols": FEATURE_COLS,
    "cv_results"  : {
        "model_a_auc"     : scores_a,
        "model_b_mae"     : scores_b,
        "model_c_mae"     : scores_c,
    },
    "params": {
        "LOOKBACK_DAYS": LOOKBACK_DAYS,
        "HORIZON_DAYS" : HORIZON_DAYS,
        "STEP_DAYS"    : STEP_DAYS,
    }
}

with open(OUTPUT_DIR / "cyano_models.pkl", "wb") as f:
    pickle.dump(model_bundle, f)

print(f"✅ Models saved → {OUTPUT_DIR / 'cyano_models.pkl'}")
print()
print("Output files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name}")


---
## 16. Load Models & Run Inference (standalone)

Use this cell to load saved models and run predictions without retraining.


In [ ]:
# with open("outputs/cyano_models.pkl", "rb") as f:
#     bundle = pickle.load(f)
#
# clf_a      = bundle["onset_clf"]
# reg_b      = bundle["duration_reg"]
# reg_c      = bundle["timing_reg"]
# FEATURE_COLS = bundle["feature_cols"]
#
# result = predict_next_6_months("Polo Lake", "2025-01-01")
# print(result)

print("Uncomment the cells above to load and run saved models.")
